# Chapter 67 — The Full Transformer Block

The previous chapters developed the main transformer components separately.

This chapter assembles causal multi-head attention, position-wise feedforward computation, layer normalization, and residual connections into one reusable block.


## Learning goals

By the end of this chapter, you will be able to:

- explain the job of each transformer-block component;
- trace the order of operations in a pre-normalized block;
- implement a complete `TransformerBlock`;
- inspect every major intermediate tensor;
- verify both residual additions;
- explain why the block preserves shape;
- test that causal attention prevents future information leakage; and
- stack several independent blocks.


## The pre-normalized block

A **transformer block** is a reusable module containing an attention sublayer and a feedforward sublayer.

A **sublayer** is one major transformation inside the block.

This chapter uses **pre-normalization**, which applies layer normalization before each sublayer transformation.

The ordered computation is:

1. Normalize the input values.
2. Apply causal multi-head attention.
3. Add the original input through a residual connection.
4. Normalize the values after attention.
5. Apply the position-wise feedforward network.
6. Add the values after attention through a second residual connection.

In compact notation:

```text
values = values + attention(layer_norm(values))
values = values + feedforward(layer_norm(values))
```


## The shape contract

Let `B` be batch size, `T` be context length, and `C` be embedding dimension.

The block receives and returns `[B, T, C]`.

| Stage | Shape |
|---|---|
| Input | `[B, T, C]` |
| First normalized values | `[B, T, C]` |
| Attention output | `[B, T, C]` |
| Values after attention residual | `[B, T, C]` |
| Second normalized values | `[B, T, C]` |
| Feedforward output | `[B, T, C]` |
| Block output | `[B, T, C]` |

The feedforward network temporarily expands only its internal feature dimension to `4C`.

Returning both sublayers to `C` makes their residual additions possible.


## Create a small input

The block receives vectors rather than raw token strings.

In a language model, these values would initially come from token embeddings plus position embeddings.

We will use a deterministic tensor with shape `[2, 4, 8]`.


In [1]:
import torch

device = "cpu"
torch.manual_seed(67)

batch_size = 2
context_length = 4
embedding_dimension = 8
number_of_heads = 4

input_values = torch.randn(
    batch_size,
    context_length,
    embedding_dimension,
    device=device,
)

print("device:", device)
print("input shape:", input_values.shape)
print("head size:", embedding_dimension // number_of_heads)

device: cpu
input shape: torch.Size([2, 4, 8])
head size: 2


The equal-head design gives each of the four heads two output features.


## Implement causal attention

The following classes reuse the explicit `ModuleList` design from Chapter 62.

Each head has separate query, key, and value projections.

The head outputs are concatenated and projected back to `C` for the residual connection.

Production implementations usually combine the per-head projections into larger matrix operations for efficiency, but the meaning is the same.


In [2]:
import math


class SingleHeadCausalSelfAttention(torch.nn.Module):
    """Compute one head of scaled causal self-attention."""

    embedding_dimension: int
    head_size: int
    context_length: int
    query_projection: torch.nn.Linear
    key_projection: torch.nn.Linear
    value_projection: torch.nn.Linear
    causal_mask: torch.Tensor

    def __init__(
        self,
        embedding_dimension: int,
        head_size: int,
        context_length: int,
    ) -> None:
        super().__init__()

        if min(embedding_dimension, head_size, context_length) <= 0:
            raise ValueError("all dimensions must be positive.")

        self.embedding_dimension = embedding_dimension
        self.head_size = head_size
        self.context_length = context_length
        self.query_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(1, context_length, context_length, dtype=torch.bool)),
        )

    def forward_with_weights(
        self, input_values: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if input_values.ndim != 3:
            raise ValueError("input must have shape [batch, context, embedding].")
        if input_values.shape[-1] != self.embedding_dimension:
            raise ValueError("input embedding dimension is incorrect.")

        current_context_length = input_values.shape[-2]
        if current_context_length > self.context_length:
            raise ValueError("input exceeds the configured context length.")

        queries = self.query_projection(input_values)
        keys = self.key_projection(input_values)
        values = self.value_projection(input_values)
        scores = queries @ keys.transpose(-2, -1)
        scaled_scores = scores / math.sqrt(self.head_size)
        allowed = self.causal_mask[:, :current_context_length, :current_context_length]
        weights = torch.softmax(
            scaled_scores.masked_fill(~allowed, float("-inf")),
            dim=-1,
        )
        return weights @ values, weights

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        output, _ = self.forward_with_weights(input_values)
        return output


class MultiHeadCausalSelfAttention(torch.nn.Module):
    """Run equal-sized causal attention heads and combine their outputs."""

    embedding_dimension: int
    number_of_heads: int
    head_size: int
    context_length: int
    attention_heads: torch.nn.ModuleList
    output_projection: torch.nn.Linear

    def __init__(
        self,
        embedding_dimension: int,
        number_of_heads: int,
        context_length: int,
    ) -> None:
        super().__init__()

        if min(embedding_dimension, number_of_heads, context_length) <= 0:
            raise ValueError("all dimensions and the head count must be positive.")
        if embedding_dimension % number_of_heads != 0:
            raise ValueError(
                "embedding dimension must divide evenly by the head count."
            )

        self.embedding_dimension = embedding_dimension
        self.number_of_heads = number_of_heads
        self.head_size = embedding_dimension // number_of_heads
        self.context_length = context_length
        self.attention_heads = torch.nn.ModuleList(
            [
                SingleHeadCausalSelfAttention(
                    embedding_dimension=embedding_dimension,
                    head_size=self.head_size,
                    context_length=context_length,
                )
                for _ in range(number_of_heads)
            ]
        )
        self.output_projection = torch.nn.Linear(
            number_of_heads * self.head_size,
            embedding_dimension,
        )

    def forward_with_weights(
        self, input_values: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        head_outputs = []
        head_weights = []

        for attention_head in self.attention_heads:
            output, weights = attention_head.forward_with_weights(input_values)
            head_outputs.append(output)
            head_weights.append(weights)

        concatenated = torch.cat(head_outputs, dim=-1)
        stacked_weights = torch.stack(head_weights, dim=1)
        projected = self.output_projection(concatenated)
        return projected, stacked_weights

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        projected, _ = self.forward_with_weights(input_values)
        return projected

The causal mask is a registered buffer, so it belongs to module state without becoming a trainable parameter.


## Implement the position-wise feedforward network

The feedforward network uses the `C → 4C → C` structure from Chapter 66.

ReLU keeps this teaching implementation easy to inspect, although many modern models use GELU or gated activations.


In [3]:
class PositionWiseFeedForward(torch.nn.Module):
    """Apply one shared feedforward network at every position."""

    embedding_dimension: int
    hidden_dimension: int
    network: torch.nn.Sequential

    def __init__(self, embedding_dimension: int, expansion_factor: int = 4) -> None:
        super().__init__()

        if embedding_dimension <= 0 or expansion_factor <= 0:
            raise ValueError(
                "embedding dimension and expansion factor must be positive."
            )

        self.embedding_dimension = embedding_dimension
        self.hidden_dimension = expansion_factor * embedding_dimension
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, self.hidden_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(self.hidden_dimension, embedding_dimension),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        if input_values.shape[-1] != self.embedding_dimension:
            raise ValueError("input embedding dimension is incorrect.")
        return self.network(input_values)

Attention can exchange information across causal positions, while this feedforward module transforms each position independently.

Both return feature dimension `C`, so both can be placed inside residual branches.


## Assemble the complete block

The block owns two separate `LayerNorm` modules because attention and feedforward receive values from different points in the computation.

The `forward_with_intermediates` method exposes the same steps as `forward` without duplicating the block logic in a separate inspection function.


In [4]:
class TransformerBlock(torch.nn.Module):
    """Apply one pre-normalized causal transformer block."""

    embedding_dimension: int
    context_length: int
    first_layer_norm: torch.nn.LayerNorm
    attention: MultiHeadCausalSelfAttention
    second_layer_norm: torch.nn.LayerNorm
    feedforward: PositionWiseFeedForward

    def __init__(
        self,
        embedding_dimension: int,
        number_of_heads: int,
        context_length: int,
    ) -> None:
        super().__init__()
        self.embedding_dimension = embedding_dimension
        self.context_length = context_length
        self.first_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_heads=number_of_heads,
            context_length=context_length,
        )
        self.second_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = PositionWiseFeedForward(embedding_dimension)

    def _validate_input(self, input_values: torch.Tensor) -> None:
        if input_values.ndim != 3:
            raise ValueError("input must have shape [batch, context, embedding].")
        if input_values.shape[-1] != self.embedding_dimension:
            raise ValueError("input embedding dimension is incorrect.")
        if input_values.shape[-2] > self.context_length:
            raise ValueError("input exceeds the configured context length.")

    def forward_with_intermediates(
        self, input_values: torch.Tensor
    ) -> dict[str, torch.Tensor]:
        self._validate_input(input_values)
        normalized_before_attention = self.first_layer_norm(input_values)
        attention_output, attention_weights = self.attention.forward_with_weights(
            normalized_before_attention
        )
        values_after_attention = input_values + attention_output
        normalized_before_feedforward = self.second_layer_norm(values_after_attention)
        feedforward_output = self.feedforward(normalized_before_feedforward)
        output_values = values_after_attention + feedforward_output

        return {
            "input_values": input_values,
            "normalized_before_attention": normalized_before_attention,
            "attention_weights": attention_weights,
            "attention_output": attention_output,
            "values_after_attention": values_after_attention,
            "normalized_before_feedforward": normalized_before_feedforward,
            "feedforward_output": feedforward_output,
            "output_values": output_values,
        }

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        return self.forward_with_intermediates(input_values)["output_values"]


torch.manual_seed(67)
transformer_block = TransformerBlock(
    embedding_dimension=embedding_dimension,
    number_of_heads=number_of_heads,
    context_length=context_length,
).to(device)

print(transformer_block)

TransformerBlock(
  (first_layer_norm): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (attention): MultiHeadCausalSelfAttention(
    (attention_heads): ModuleList(
      (0-3): 4 x SingleHeadCausalSelfAttention(
        (query_projection): Linear(in_features=8, out_features=2, bias=False)
        (key_projection): Linear(in_features=8, out_features=2, bias=False)
        (value_projection): Linear(in_features=8, out_features=2, bias=False)
      )
    )
    (output_projection): Linear(in_features=8, out_features=8, bias=True)
  )
  (second_layer_norm): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (feedforward): PositionWiseFeedForward(
    (network): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=8, bias=True)
    )
  )
)


This teaching block deliberately omits dropout and uses explicit per-head modules.

Those simplifications keep the data flow visible without changing the pre-normalized residual structure being studied.


## Inspect every stage

The next cell runs the block once and prints the complete external shape contract.

Attention weights are the one intentional exception because they have one square table per batch item and head.


In [5]:
inspection = transformer_block.forward_with_intermediates(input_values)
forward_output = transformer_block(input_values)

print("stage".ljust(38), "shape")
print("-" * 62)
for stage_name, stage_values in inspection.items():
    print(stage_name.ljust(38), tuple(stage_values.shape))

print()
print("forward output matches inspection:", end=" ")
print(torch.allclose(forward_output, inspection["output_values"]))

stage                                  shape
--------------------------------------------------------------
input_values                           (2, 4, 8)
normalized_before_attention            (2, 4, 8)
attention_weights                      (2, 4, 4, 4)
attention_output                       (2, 4, 8)
values_after_attention                 (2, 4, 8)
normalized_before_feedforward          (2, 4, 8)
feedforward_output                     (2, 4, 8)
output_values                          (2, 4, 8)

forward output matches inspection: True


Every value tensor at a sublayer boundary has shape `[2, 4, 8]`.

The attention-weight tensor has shape `[B, H, T, T] = [2, 4, 4, 4]`.


## Verify both residual connections

A residual connection performs elementwise addition, not concatenation.

These checks reconstruct both additions from the inspected tensors.


In [6]:
expected_after_attention = inspection["input_values"] + inspection["attention_output"]
expected_output = (
    inspection["values_after_attention"] + inspection["feedforward_output"]
)

print("attention residual shapes match:", end=" ")
print(inspection["input_values"].shape == inspection["attention_output"].shape)
print("first residual arithmetic matches:", end=" ")
print(torch.allclose(expected_after_attention, inspection["values_after_attention"]))
print("feedforward residual shapes match:", end=" ")
print(
    inspection["values_after_attention"].shape == inspection["feedforward_output"].shape
)
print("second residual arithmetic matches:", end=" ")
print(torch.allclose(expected_output, inspection["output_values"]))

attention residual shapes match: True
first residual arithmetic matches: True
feedforward residual shapes match: True
second residual arithmetic matches: True


Both transformations compute a learned change with the same shape as the values on their skip paths.

If either transformation stopped at a different feature dimension, its residual addition would be invalid.


## Verify causal behavior end to end

A shape check cannot detect information leaking from future positions.

We will change only the final input position and compare the complete block outputs.

Outputs at positions `0`, `1`, and `2` must remain unchanged because they cannot attend to position `3`.


In [7]:
changed_future_input = input_values.clone()
changed_future_input[:, -1] += 100.0 * torch.randn_like(changed_future_input[:, -1])

original_output = transformer_block(input_values)
changed_output = transformer_block(changed_future_input)
output_change_by_position = (changed_output - original_output).abs().sum(dim=-1)

print("absolute output change by position:")
print(output_change_by_position)
print("earlier positions unchanged:", end=" ")
print(torch.allclose(original_output[:, :-1], changed_output[:, :-1]))
print("final position changed:", end=" ")
print(not torch.allclose(original_output[:, -1], changed_output[:, -1]))

absolute output change by position:
tensor([[  0.0000,   0.0000,   0.0000, 703.9511],
        [  0.0000,   0.0000,   0.0000, 710.7341]], grad_fn=<SumBackward1>)
earlier positions unchanged: True
final position changed: True


Only the final output position changes.

The result depends on both properties established earlier: attention is causal, and the feedforward network does not mix positions.


## Separate normalization modules

The two layer normalizations have the same parameter shapes but are different module instances.

Training can therefore learn different affine scales and shifts before attention and before feedforward.


In [8]:
print("same LayerNorm object:", end=" ")
print(transformer_block.first_layer_norm is transformer_block.second_layer_norm)

for layer_name, layer_norm in [
    ("first", transformer_block.first_layer_norm),
    ("second", transformer_block.second_layer_norm),
]:
    print(f"{layer_name} LayerNorm parameter shapes:")
    for parameter_name, parameter in layer_norm.named_parameters():
        print(f"  {parameter_name}: {tuple(parameter.shape)}")

same LayerNorm object: False
first LayerNorm parameter shapes:
  weight: (8,)
  bias: (8,)
second LayerNorm parameter shapes:
  weight: (8,)
  bias: (8,)


`False` confirms that the modules do not share learned parameters.

Both still normalize each token vector across its eight features.


## Parameters and buffers

Trainable parameters include both layer normalizations, all attention projections, and both feedforward linear layers.

The causal masks appear as buffers instead of parameters.


In [9]:
def count_parameters(module: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters())


print("trainable parameters:", count_parameters(transformer_block))
print("parameter tensors:", len(list(transformer_block.parameters())))
print("named buffers:")
for buffer_name, buffer in transformer_block.named_buffers():
    print(f"  {buffer_name}: {tuple(buffer.shape)}")

trainable parameters: 848
parameter tensors: 22
named buffers:
  attention.attention_heads.0.causal_mask: (1, 4, 4)
  attention.attention_heads.1.causal_mask: (1, 4, 4)
  attention.attention_heads.2.causal_mask: (1, 4, 4)
  attention.attention_heads.3.causal_mask: (1, 4, 4)


Each head owns one `[1, T, T]` causal-mask buffer.

Buffers move with the module and are stored in its state, but gradient descent does not update them.


## Stack independent blocks

A shape-preserving block can feed its output directly into another block.

The stack below uses three separately initialized blocks rather than applying one block repeatedly.


In [10]:
torch.manual_seed(67)
number_of_layers = 3
transformer_blocks = torch.nn.ModuleList(
    [
        TransformerBlock(
            embedding_dimension=embedding_dimension,
            number_of_heads=number_of_heads,
            context_length=context_length,
        )
        for _ in range(number_of_layers)
    ]
).to(device)

current_values = input_values
print("initial shape:", tuple(current_values.shape))
for block_index, block in enumerate(transformer_blocks):
    current_values = block(current_values)
    print(f"after block {block_index}: {tuple(current_values.shape)}")

print("blocks are distinct objects:", end=" ")
print(transformer_blocks[0] is not transformer_blocks[1])

initial shape: (2, 4, 8)
after block 0: (2, 4, 8)
after block 1: (2, 4, 8)
after block 2: (2, 4, 8)
blocks are distinct objects: True


Every block preserves `[B, T, C]`, so depth changes the values without changing the interface between blocks.

Each block has its own learned parameters and can develop a different transformation.


## Pre-norm and post-norm are different

This chapter implements pre-normalization:

```text
values = values + sublayer(layer_norm(values))
```

A post-normalized design instead uses this order:

```text
values = layer_norm(values + sublayer(values))
```

The two designs can preserve the same shape, but they compute different values and have different optimization behavior.

Pre-normalization is common in modern GPT-style architectures and is the course convention here, but post-normalized transformers also exist.


## The block is not the whole language model

This transformer block accepts and returns hidden vectors.

A complete causal language model also needs:

- token and position embeddings before the block stack;
- a final layer normalization after the stack;
- an output projection from hidden vectors to vocabulary logits; and
- training and generation logic.

Those model-level pieces belong outside each repeated block.


## Common mistakes

- Do not move layer normalization after the residual addition and still call the block pre-normalized.
- Do not reuse one `LayerNorm` object for both normalization points.
- Do not omit the attention output projection required to return to `C`.
- Do not leave the feedforward output at hidden dimension `4C`.
- Do not remove causal masking from a GPT-style attention sublayer.
- Do not claim that the position-wise feedforward network mixes positions.
- Do not confuse stacking independent blocks with repeatedly calling one shared block.


## Takeaways

A complete pre-normalized transformer block performs two normalized residual updates.

```text
values = values + attention(layer_norm(values))
values = values + feedforward(layer_norm(values))
```

Attention exchanges information across causally allowed positions.

The feedforward network then transforms each position independently.

Both sublayers return `[B, T, C]`, which makes residual addition and block stacking possible.

The order is part of the architecture, not a cosmetic choice.


## What comes next

The next chapter will place token and position embeddings before a stack of these blocks, then add final normalization and a vocabulary projection.

That assembly turns the isolated transformer block into a small GPT-style language model.
